1. Importar librerías

In [ ]:
# ==========================================
# 1. Importar librerías
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)


2. Cargar dataset limpio

In [ ]:
# ==========================================
# 2. Cargar dataset limpio
# ==========================================
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
file_path = os.path.join(project_root, "data/processed/online_news_cleaned.csv")

df = pd.read_csv(file_path)
df.head()



# 🟧 Grupo C — Keywords (`kw_*`)
Estas 9 variables son derivadas del análisis de “keywords” (palabras clave) asociadas a cada artículo.  
No representan “número de palabras” sino **popularidad histórica** de esas keywords dentro del dataset de Mashable.

> En otras palabras:  
> **Cada keyword tiene un nivel de popularidad global**, calculado a partir del total de shares que esa palabra ha tenido en otros artículos.  
>  
> Las variables `kw_*` miden cómo son, en promedio, las keywords asociadas a un artículo.

---

## 📌 ¿Qué mide exactamente cada variable?

Mashable asignaba a cada keyword un “score” basado en:
- número de shares históricos,
- relevancia dentro de Mashable,
- frecuencia de aparición,
- impacto en tráfico previo.

El dataset solo incluye los estadísticos finales, no la fórmula interna.

A continuación se explica las 9 variables una por una.

---

# 🟧 Explicación variable por variable

## 1) `kw_min_min`
🟡 **Definición:**  
El **valor mínimo** entre todos los *mínimos de popularidad* de las keywords asignadas al artículo.

👉 Ejemplo:  
Si un artículo tiene keywords con popularidad mínima:  
- 100  
- 150  
- 30  

Entonces `kw_min_min = 30`.

🟢 **Valores típicos:**  
0 a 200  
(99% de las veces es pequeño).

🧠 **Interpretación:**  
Indica si el artículo incluye alguna keyword muy poco popular.

---

## 2) `kw_max_min`
🟡 **Definición:**  
El **valor máximo** dentro de los *mínimos de popularidad* de las keywords del artículo.

👉 Ejemplo:  
Si las keywords tienen popularidades mínimas:  
- 100  
- 150  
- 30  

`kw_max_min = 150`.

🧠 **Interpretación:**  
Mide si el artículo tiene keywords con una base mínima muy fuerte (keywords donde incluso “su punto más bajo” es elevado).

🟢 **Valores típicos:** 1000 – 20,000  
Muy variable.

---

## 3) `kw_avg_min`
🟡 **Definición:**  
Promedio de los *valores mínimos* de popularidad para cada keyword del artículo.

🧠 **Interpretación:**  
Mide si, en general, el artículo usa keywords cuya popularidad mínima es estable y consistente.

🟢 **Valores esperados:** rangos entre 200 y 5,000.

---

## 4) `kw_min_max`
🟡 **Definición:**  
El **valor mínimo** entre todos los *máximos históricos* de popularidad de cada keyword.

🧠 **Interpretación:**  
Si este valor es grande → todas las keywords tienen un tope alto de popularidad histórica.  
Si es pequeño → al menos una keyword jamás fue muy popular.

🟢 **Valores típicos:** 500 – 50,000  
(sesgado por outliers).

---

## 5) `kw_avg_max`
🟡 **Definición:**  
Promedio de los valores máximos de popularidad de las keywords.

🧠 **Interpretación:**  
Captura si el artículo contiene keywords que alguna vez fueron muy populares (hits virales).

🟢 **Valores típicos:** 5,000 – 30,000  
Depende de la temática del artículo.

---

## 6) `kw_max_avg`
🟡 **Definición:**  
Máximo de los *promedios históricos* de popularidad de las keywords del artículo.

🧠 **Interpretación:**  
Mide si el artículo contiene **una keyword fuerte**, con alto promedio histórico (no solo momentos aislados).

🟢 **Valores esperados:** 8,000 – 40,000.

---

## 7) `kw_min_avg`
🟡 **Definición:**  
El valor mínimo de los promedios históricos de popularidad de sus keywords.

🧠 **Interpretación:**  
Revela si alguna keyword es, en promedio, muy débil (lo cual puede reducir el impacto del artículo).

🟢 **Valores típicos:** 100 – 3,000.

---

## 8) `kw_max_avg`
🟡 **Definición:**  
Máximo valor del promedio de popularidad entre las palabras clave.

🧠 **Interpretación:**  
Señala si hay **una keyword consistentemente viral**.

🟢 **Valores típicos:** 10,000 – 50,000.

---

## 9) `kw_avg_avg`
🟡 **Definición:**  
Promedio de los promedios históricos de popularidad de las keywords.

🧠 **Interpretación:**  
Representa la **popularidad promedio general** de todas las keywords del artículo.

Es la más estable y la más útil del grupo.

🟢 **Valores típicos:** 2,000 – 12,000.

---

# 🟧 ¿Por qué estos valores tienen escalas tan grandes?

Porque Mashable usa una escala acumulativa basada en **total de shares históricos**, que puede ir desde:
- palabras muy raras → 10–200  
- palabras de moda → 20,000–80,000  
- palabras hiper-populares (“Facebook”, “iPhone”, “Google”) → 100,000+

El dataset recorta algunos valores, pero conserva la escala grande.

---

# 🟧 Sobre los valores -1

Cuando Mashable no encuentra ninguna keyword en un artículo:
- todas las variables se llenan con `-1`
- lo correcto es reemplazarlo por `NaN`
- significa: **no hay información de keywords para este artículo**

Estos artículos típicamente:
- tienen menor viralidad,
- están en categorías raras,
- o son notas muy cortas (poca riqueza semántica).

---

# 🟧 Resumen de interpretación de las keywords

| Variable | ¿Qué mide? | ¿Qué significa un valor alto? |
|---------|-------------|-------------------------------|
| kw_min_min | Keyword más débil | Al menos hay una keyword poco popular |
| kw_max_min | El mejor mínimo | Todas las keywords son relativamente fuertes |
| kw_avg_min | Promedio de mínimos | Estabilidad general de popularidad baja |
| kw_min_max | El peor máximo | Alguna keyword nunca fue popular |
| kw_avg_max | Promedio de máximos | Historial de picos de popularidad |
| kw_max_avg | Mayor promedio | Una keyword estrella muy estable |
| kw_min_avg | Peor promedio | Al menos una keyword floja |
| kw_max_avg | Mejor promedio | Keyword consistentemente viral |
| kw_avg_avg | Promedio global | Popularidad general de las keywords del artículo |

---
# 📘 ¿Qué es *Mashable*?

**Mashable** es un medio digital estadounidense fundado en 2005 por Pete Cashmore.

Es un sitio web enfocado en:
- tecnología,
- redes sociales,
- entretenimiento,
- cultura digital,
- tendencias virales,
- innovación y negocios digitales.

Mashable se caracteriza por publicar artículos que suelen viralizarse rápidamente debido a su enfoque en:
- plataformas sociales,
- noticias de última hora,
- contenido orientado a audiencias jóvenes,
- cultura de internet.

También es conocido por su influencia en temas de:
- marketing digital,
- social media,
- startups,
- gadgets,
- análisis de tendencias online.

En el dataset de **Online News Popularity**, *Mashable* es **la fuente original** de todas las noticias analizadas, y la variable `shares` corresponde al número de veces que cada artículo fue compartido en redes sociales.


In [ ]:
[col for col in kw_vars if col not in df.columns]


In [ ]:
# ==========================================
# 🟧 PREPARACIÓN — Grupo C: Keywords (kw_*)
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

sns.set(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

# ============================
# Lista correcta de variables kw_*
# ============================
kw_vars = [
    "kw_min_min",
    "kw_max_min",
    "kw_avg_min",
    "kw_min_max",
    "kw_max_max",    
    "kw_avg_max",
    "kw_min_avg",
    "kw_max_avg",
    "kw_avg_avg"
]

# ============================
# Reemplazar -1 por NaN
# ============================
df[kw_vars] = df[kw_vars].replace(-1, np.nan)

df[kw_vars].head()


✅ 1️⃣ VIOLINPLOTS — Distribuciones avanzadas

In [ ]:
# ==========================================
# 1. VIOLINPLOTS — Distribuciones avanzadas
# ==========================================

# "Melt" para graficación
kw_melt = df[kw_vars].melt(var_name="variable", value_name="value")

plt.figure(figsize=(12, 6))
sns.violinplot(data=kw_melt, x="variable", y="value", inner="quartile")
plt.xticks(rotation=45)
plt.title("Distribución de variables kw_* (violinplot)")
plt.tight_layout()
plt.show()

# ==========================================
# Versión log-transformada (para colas pesadas)
# ==========================================

kw_melt_log = df[kw_vars].apply(np.log1p).melt(var_name="variable", value_name="value_log")

plt.figure(figsize=(12, 6))
sns.violinplot(data=kw_melt_log, x="variable", y="value_log", inner="quartile")
plt.xticks(rotation=45)
plt.title("Distribución log-transformada de variables kw_* (violinplot)")
plt.tight_layout()
plt.show()


✅ 2️⃣ POWER-LAW PLOTS — Colas pesadas

In [ ]:
# ==========================================
# 2. POWER-LAW PLOTS — Colas pesadas
# ==========================================

for col in kw_vars:
    series = df[col].dropna().sort_values(ascending=False)
    ranks = np.arange(1, len(series) + 1)

    plt.figure(figsize=(6, 4))
    plt.loglog(ranks, series.values, marker=".", linestyle="none")
    plt.xlabel("Rango (log)")
    plt.ylabel(f"{col} (log)")
    plt.title(f"Power-law plot de {col}")
    plt.grid(True, which="both", ls="--", linewidth=0.5)
    plt.tight_layout()
    plt.show()


✅ 3️⃣ PCA COMPLETO

In [ ]:
# ==========================================
# 3. PCA — Preparación de datos
# ==========================================

# Imputación simple (puedes usar medianas si deseas)
kw_imputed = df[kw_vars].fillna(0)

# Escalar
scaler = StandardScaler()
kw_scaled = scaler.fit_transform(kw_imputed)

# DataFrame escalado (opcional)
kw_scaled_df = pd.DataFrame(kw_scaled, columns=kw_vars)
kw_scaled_df.head()


3.2 Ejecutar PCA y analizar varianza explicada

In [ ]:
# ==========================================
# 3.2 PCA — Ajuste e interpretación
# ==========================================

pca = PCA(n_components=3)
kw_pca = pca.fit_transform(kw_scaled)

explained_var = pca.explained_variance_ratio_

# Tabla de varianza explicada
pca_var_df = pd.DataFrame({
    "Componente": [f"PC{i+1}" for i in range(len(explained_var))],
    "Varianza_explicada": explained_var,
    "Varianza_acumulada": np.cumsum(explained_var)
})
pca_var_df


3.3 Loadings (contribuciones)

In [ ]:
# ==========================================
# 3.3 PCA — Loadings
# ==========================================

loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(pca.n_components_)],
    index=kw_vars
)

loadings

# Orden de importancia por componente
for pc in loadings.columns:
    print(f"\n🔎 Variables que más contribuyen a {pc}:")
    display(loadings[pc].sort_values(ascending=False))


3.4 Agregar componentes al DataFrame

In [ ]:
df["kw_PC1"] = kw_pca[:, 0]
df["kw_PC2"] = kw_pca[:, 1]
df["kw_PC3"] = kw_pca[:, 2]

df[["kw_PC1", "kw_PC2", "kw_PC3"]].head()


✅ 4️⃣ FEATURE ENGINEERING OPTIMIZADO

In [ ]:
# ==========================================
# 4. FEATURE ENGINEERING — Grupo C
# ==========================================

# 4.1 Log-transform de cada variable keyword
for col in kw_vars:
    df[f"log1p_{col}"] = np.log1p(df[col])

# 4.2 Selección de features clave
kw_core = [
    "kw_avg_max",
    "kw_max_avg",
    "kw_avg_avg"
]

df[kw_core].describe().T


✅ 5️⃣ SCORING FINAL — Keyword Strength

In [ ]:
# ==========================================
# 5. SCORING — Keyword Strength
# ==========================================

kw_strength_features = ["kw_avg_max", "kw_max_avg", "kw_avg_avg"]

# Imputación
kw_strength_imputed = df[kw_strength_features].fillna(0)

# Escalar a [0,1]
mm_scaler = MinMaxScaler()
kw_strength_scaled = mm_scaler.fit_transform(kw_strength_imputed)

kw_strength_scaled_df = pd.DataFrame(
    kw_strength_scaled,
    columns=[f"{c}_norm" for c in kw_strength_features]
)

# Agregar al DataFrame original
for c in kw_strength_scaled_df.columns:
    df[c] = kw_strength_scaled_df[c]

# Score final con pesos
df["keyword_strength_score"] = (
    0.4 * df["kw_avg_avg_norm"] +
    0.3 * df["kw_avg_max_norm"] +
    0.3 * df["kw_max_avg_norm"]
)

df[["kw_avg_max", "kw_max_avg", "kw_avg_avg", "keyword_strength_score"]].head()


✅ Visualización del score vs viralidad

In [ ]:
# ==========================================
# Crear variable objetivo transformada
# ==========================================
df["log_shares"] = np.log1p(df["shares"])


In [ ]:
plt.figure(figsize=(6,4))
sns.scatterplot(x=df["keyword_strength_score"], y=df["log_shares"], alpha=0.3)
plt.title("Keyword Strength Score vs log_shares")
plt.show()
